# CampusResolve AI: Step 1 to 3 baseline (runs locally, no AWS needed)

Trains three text models on the **real** complaint data and evaluates them on the real, untouched test set:

1. **Category** (Academic, Administrative, Finance, Infrastructure, Technical)
2. **Priority** (Low, Medium, High, Urgent)
3. **Department** (routing)

Then adds a **safety-rule layer** and a **low-confidence flag** (human review), and saves the models as `.joblib` files that you can upload to S3 later.

**Rules:** never train on `test.csv`; never add synthetic rows to it.

> In Google Colab: run the next cell and upload `train.csv` and `test.csv`. On a laptop, put both files next to this notebook.

In [1]:
import os
try:
    from google.colab import files
    if not os.path.exists('train.csv'):
        files.upload()   # choose train.csv and test.csv
except ImportError:
    pass

import re, json, joblib
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import make_pipeline, make_union
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

train = pd.read_csv('train.csv'); test = pd.read_csv('test.csv')
TEXT = 'Complaint_Description'
print(train.shape, test.shape)
train.head(3)

(273, 12) (59, 12)


,ID,Timestamp,Gender,Semester,Student_Dept,Complaint_Description,Category,Responsible_Departments,Primary_Department,Aspects,Severity,Complaint_Group_ID
0,1,2026-02-09 12:18:58.314,Male,4.1,IPE,"On most parts of the 8th/9th floor, WiFi is no...",Technical,IT,IT,WiFi | Lab Equipment,Urgent,0
1,4,2026-02-09 12:28:16.321,Female,4.1,CSE,need a seperate female prayer room which is no...,Infrastructure,Student Welfare | Administration | Maintenance,Student Welfare,Classroom | Prayer Room,Medium,3
2,5,2026-02-09 12:29:14.893,Male,4.1,CSE,drinking water system on the 6th floor(CSE dpt...,Infrastructure,Maintenance,Maintenance,Water Supply,Urgent,4


## 1. Data checks (leakage and class balance)

In [2]:
# Group IDs mark near-duplicate complaints. If a group appears in BOTH files, scores would be inflated.
overlap = set(train.Complaint_Group_ID) & set(test.Complaint_Group_ID)
print('Groups in both train and test:', len(overlap), '(should be 0)')
print('Exact text overlap:', len(set(train[TEXT]) & set(test[TEXT])), '(should be 0)')
print('Duplicate texts inside train:', train[TEXT].duplicated().sum())
for col in ['Category','Severity','Primary_Department']:
    print('\n', col); print(pd.concat([train[col].value_counts().rename('train'), test[col].value_counts().rename('test')], axis=1).fillna(0).astype(int))

Groups in both train and test: 0 (should be 0)
Exact text overlap: 0 (should be 0)
Duplicate texts inside train: 5

 Category
                train  test
Category                   
Infrastructure    114    29
Technical          58     7
Academic           48    16
Finance            29     4
Administrative     24     3

 Severity
          train  test
Severity             
Urgent       97    11
Medium       79    20
High         75    23
Low          22     5

 Primary_Department
                     train  test
Primary_Department              
Maintenance             79    19
IT                      56    13
Department Head         52     8
Transport & Parking     21     4
Finance                 21     4
Student Welfare         17     3
Library                 12     0
Proctorial               8     0
Registrar                4     0
Canteen                  3     0
Examination              0     8


## 2. Train the three models

Features: word 1-2 grams + character 3-5 grams (character n-grams help with typos).
Two models per task: a plain **Linear SVM** decides the label (most accurate here), and a **calibrated copy** only supplies the confidence score.
`class_weight='balanced'` helps rare classes such as Low severity.

In [3]:
def make_feats():
    return make_union(
        TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True),
        TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), sublinear_tf=True))

def build_label_model():      # decides the label (most accurate on this data)
    return make_pipeline(make_feats(), LinearSVC(C=1, class_weight='balanced'))

def build_conf_model():       # only used to estimate confidence (calibrated probabilities)
    return make_pipeline(make_feats(), CalibratedClassifierCV(LinearSVC(C=1, class_weight='balanced'), cv=3))

TARGETS = {'category':'Category', 'priority':'Severity', 'department':'Primary_Department'}
models, conf_models = {}, {}
for name, col in TARGETS.items():
    models[name] = build_label_model().fit(train[TEXT], train[col])
    conf_models[name] = build_conf_model().fit(train[TEXT], train[col])
    print('trained', name)

def predict_with_conf(name, texts):
    # label from the plain SVM; confidence = calibrated probability of that same label
    labels = models[name].predict(texts)
    proba = conf_models[name].predict_proba(texts)
    idx = {c:i for i,c in enumerate(conf_models[name].classes_)}
    conf = np.array([proba[r, idx[l]] for r,l in enumerate(labels)])
    return labels, conf

trained category


trained priority


trained department


## 3. Honest evaluation on the real test set

In [4]:
preds = {}
rows = []
for name, col in TARGETS.items():
    p = models[name].predict(test[TEXT]); preds[name] = p
    rows.append({'task':name, 'accuracy':accuracy_score(test[col], p), 'macro_F1':f1_score(test[col], p, average='macro')})
pd.DataFrame(rows).round(3)

,task,accuracy,macro_F1
0,category,0.932,0.886
1,priority,0.746,0.590
2,department,0.847,0.763


In [5]:
# Is the priority model learning from TEXT, or just guessing from category? Compare with trivial baselines.
maj = train.Severity.mode()[0]
cat_lookup = train.groupby('Category').Severity.agg(lambda s: s.mode()[0])
print('Always predict most common severity :', round((test.Severity==maj).mean(),3))
print('Severity looked up from true category:', round((test.Category.map(cat_lookup)==test.Severity).mean(),3))
print('Our priority model                   :', round(accuracy_score(test.Severity, preds['priority']),3))

Always predict most common severity : 0.186
Severity looked up from true category: 0.288
Our priority model                   : 0.746


In [6]:
print(classification_report(test.Category, preds['category'], zero_division=0))
print(classification_report(test.Severity, preds['priority'], zero_division=0))

                precision    recall  f1-score   support

      Academic       1.00      0.75      0.86        16
Administrative       0.60      1.00      0.75         3
       Finance       0.80      1.00      0.89         4
Infrastructure       1.00      1.00      1.00        29
     Technical       0.88      1.00      0.93         7

      accuracy                           0.93        59
     macro avg       0.86      0.95      0.89        59
  weighted avg       0.95      0.93      0.93        59

              precision    recall  f1-score   support

        High       0.76      0.70      0.73        23
         Low       0.00      0.00      0.00         5
      Medium       0.72      0.90      0.80        20
      Urgent       0.77      0.91      0.83        11

    accuracy                           0.75        59
   macro avg       0.56      0.63      0.59        59
weighted avg       0.68      0.75      0.71        59



### Data issue to know about: the `Examination` department
`Examination` appears **8 times in test.csv but never in train.csv**, so no model can learn it from this data. That caps department accuracy at about 86% and explains most Academic mistakes. The fix is more training examples for it (for example Bedrock-generated exam-related complaints reviewed by you), not a different algorithm.

In [7]:
print('Departments in test but not in train:', set(test.Primary_Department) - set(train.Primary_Department))
seen = test.Primary_Department.isin(train.Primary_Department)
print('Department accuracy on rows whose department exists in train:', round(accuracy_score(test.Primary_Department[seen], preds['department'][seen]),3), f'({seen.sum()} rows)')

Departments in test but not in train: {'Examination'}
Department accuracy on rows whose department exists in train: 0.98 (51 rows)


In [8]:
labels = ['Low','Medium','High','Urgent']
print('Priority confusion matrix (rows=true, cols=predicted)')
print(pd.DataFrame(confusion_matrix(test.Severity, preds['priority'], labels=labels), index=labels, columns=labels))

Priority confusion matrix (rows=true, cols=predicted)
        Low  Medium  High  Urgent
Low       0       2     2       1
Medium    0      18     2       0
High      0       5    16       2
Urgent    0       0     1      10


## 4. Safety-rule layer

ML alone should never decide whether a safety-critical complaint is urgent. These rules can only **raise** priority, never lower it.
Edit the keyword lists to match your campus.

In [9]:
URGENT_PATTERNS = [
    r'\bfire\b', r'\bsmoke\b', r'short[- ]?circuit', r'electric(al)? shock', r'\bsparks?\b', r'exposed wires?',
    r'gas leak', r'\bragging\b', r'harass', r'assault', r'molest', r'\bthreat', r'suicid', r'self[- ]?harm',
    r'ambulance', r'medical emergency', r'\bunconscious\b', r'stuck in (the )?(lift|elevator)', r'\bsnake\b',
    r'collaps', r'stampede']
HIGH_PATTERNS = [r'deadline (is )?(today|tomorrow)', r'exam (is )?(today|tomorrow)', r'money (was )?deducted', r'payment (was )?deducted',
                 r'\bleak(ing|age)?\b.*\b(roof|ceiling)\b', r'broken (glass|handrail)', r'no (drinking )?water']
ORDER = {'Low':0,'Medium':1,'High':2,'Urgent':3}
INV = {v:k for k,v in ORDER.items()}

def apply_safety_rules(text, ml_priority):
    t = text.lower()
    for pat in URGENT_PATTERNS:
        if re.search(pat, t): return 'Urgent', 'rule:'+pat
    for pat in HIGH_PATTERNS:
        if re.search(pat, t) and ORDER[ml_priority] < ORDER['High']: return 'High', 'rule:'+pat
    return ml_priority, None

# does it ever fire on the real test set? (it should rarely change anything)
changed = [(t, p, *apply_safety_rules(t, p)) for t, p in zip(test[TEXT], preds['priority'])]
print('Test complaints changed by rules:', sum(1 for c in changed if c[1]!=c[2]))
for t,p,new,why in changed:
    if p!=new: print(' -', t[:80], '|', p, '->', new, why)

Test complaints changed by rules: 0


## 5. Confidence threshold (human review)

If the model is not confident, do not auto-decide: send the ticket to staff.
The table shows the trade-off: a higher threshold means more tickets go to humans but the auto-handled ones are more accurate.

In [10]:
_, cat_proba = predict_with_conf('category', test[TEXT])
_, pri_proba = predict_with_conf('priority', test[TEXT])
out = []
for th in [0.0, 0.4, 0.5, 0.6, 0.7]:
    ok = (cat_proba>=th) & (pri_proba>=th)
    out.append({'threshold':th, 'auto-handled %':round(ok.mean()*100,1),
                'category acc (auto)':round(accuracy_score(test.Category[ok], preds['category'][ok]),3),
                'priority acc (auto)':round(accuracy_score(test.Severity[ok], preds['priority'][ok]),3)})
pd.DataFrame(out)

,threshold,auto-handled %,category acc (auto),priority acc (auto)
0,0.0,100.0,0.932,0.746
1,0.4,64.4,0.974,0.763
2,0.5,45.8,1.000,0.778
3,0.6,33.9,1.000,0.850
4,0.7,8.5,1.000,1.000


In [11]:
REVIEW_THRESHOLD = 0.5   # pick from the table above; revisit when you have more data

def triage(text):
    """Full pipeline for one complaint. This function is what Lambda / SageMaker will run later."""
    s = pd.Series([text])
    out = {}
    for name in TARGETS:
        label, conf = predict_with_conf(name, s)
        out[name] = label[0]; out[name+'_conf'] = round(float(conf[0]), 3)
    out['priority_ml'] = out['priority']
    out['priority'], rule = apply_safety_rules(text, out['priority'])
    out['rule_fired'] = rule
    out['needs_human_review'] = bool(out['category_conf'] < REVIEW_THRESHOLD or out['priority_conf'] < REVIEW_THRESHOLD or out['department_conf'] < REVIEW_THRESHOLD)
    return out

for t in ["Wifi in B block is dead since yesterday",
          "There are sparks coming from the socket in lab 3",
          "My semester fee was deducted twice, receipt not generated",
          "need more charging ports in library",
          "someone is ragging juniors in hostel at night",
          "asdf hello"]:
    print(t); print('  ', triage(t))

Wifi in B block is dead since yesterday


   {'category': 'Technical', 'category_conf': 0.561, 'priority': 'Urgent', 'priority_conf': 0.601, 'department': 'IT', 'department_conf': 0.493, 'priority_ml': 'Urgent', 'rule_fired': None, 'needs_human_review': True}
There are sparks coming from the socket in lab 3
   {'category': 'Infrastructure', 'category_conf': 0.874, 'priority': 'Urgent', 'priority_conf': 0.551, 'department': 'Maintenance', 'department_conf': 0.805, 'priority_ml': 'Urgent', 'rule_fired': 'rule:\\bsparks?\\b', 'needs_human_review': False}
My semester fee was deducted twice, receipt not generated
   {'category': 'Finance', 'category_conf': 0.694, 'priority': 'Urgent', 'priority_conf': 0.422, 'department': 'Finance', 'department_conf': 0.753, 'priority_ml': 'Urgent', 'rule_fired': None, 'needs_human_review': True}
need more charging ports in library
   {'category': 'Infrastructure', 'category_conf': 0.923, 'priority': 'Medium', 'priority_conf': 0.871, 'department': 'Library', 'department_conf': 0.62, 'priority_ml': 

## 6. Save the models (these files go to S3 later)

In [12]:
os.makedirs('models', exist_ok=True)
for name in TARGETS:
    joblib.dump(models[name], f'models/{name}_model.joblib')
    joblib.dump(conf_models[name], f'models/{name}_confidence.joblib')
json.dump({'urgent_patterns':URGENT_PATTERNS, 'high_patterns':HIGH_PATTERNS, 'review_threshold':REVIEW_THRESHOLD},
          open('models/config.json','w'), indent=2)
print(os.listdir('models'))
# sanity check: reload from disk and predict
reloaded = joblib.load('models/category_model.joblib')
print(reloaded.predict(['projector is very blurry']))

['priority_confidence.joblib', 'category_model.joblib', 'priority_model.joblib', 'department_model.joblib', 'department_confidence.joblib', 'category_confidence.joblib', 'config.json']
['Infrastructure']


## Next steps
1. Read the weak spots above (Low severity has very few training rows; Academic recall is lowest).
2. When Bedrock access is approved, run `augment_with_bedrock.py`, focusing on **Low and Medium severity** and **Academic**, retrain here, and compare against these numbers on the **same** test set.
3. Upload the `models/` folder to S3, then deploy it behind Lambda or a SageMaker endpoint.